# **Compilação de legislação**
Este notebook deve ser utilizado na fase **"Concepção e Planejamento"** de projetos de desenvolvimento de assistentes e agentes virtuais de inteligência artificial generativa (IAGen), na etapa **"Curadoria dos Dados"**, com o objetivo de criar arquivos `TXT` de legislações sem dispositivos revogados (tachados no texto), obtidas a partir do *site* do Planalto.


## **1. Configuração do Ambiente**

### 1.1. Bibliotecas utilizadas

In [20]:
import os
import re
import csv
import unicodedata
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib.parse import urlparse
from urllib3.util.retry import Retry
from datetime import datetime
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from bs4.element import Tag

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 22, Finished, Available, Finished, False)

### 1.2. Funções

#### 1.2.1. Funções gerais de tratamento de dados

In [21]:
# Crie uma função para limpar sinais gráficos e caracteres especiais
# deixando todas as palavras em minúsculo

def limpar_string_lower(texto_entrada):
    """
    Higieniza a string para criação de nomes de arquivos ou identificadores.
    Remove acentos, caracteres especiais e padroniza o formato.
    """
    if not isinstance(texto_entrada, str):
        return texto_entrada

    # 1. Remove acentos (Normaliza para decompor caracteres acentuados)
    forma_nfkd = unicodedata.normalize('NFKD', texto_entrada)
    texto = "".join([c for c in forma_nfkd if not unicodedata.combining(c)])

    # 2. Converte para minúsculas (Recomendado para sistemas de arquivos no Fabric/Lakehouse)
    texto = texto.lower()

    # 3. Substitui espaços e hifens por sublinhados (undercore)
    texto = re.sub(r'[\s\-]+', '_', texto)

    # 4. Remove caracteres especiais restantes (mantém apenas a-z, 0-9 e _)
    texto = re.sub(r'[^a-z0-9_]', '', texto)

    # Retorna o texto removendo sublinhados extras no início ou fim
    return texto.strip('_')

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 23, Finished, Available, Finished, False)

In [22]:
# Crie uma função para limpar sinais gráficos e caracteres especiais
# deixando em maiúsculo somente a primeira letra de cada palavra

def limpar_string_title(texto_entrada):
    """
    Higieniza a string mantendo a primeira letra de cada palavra em maiúscula
    e preserva o hífen (ex: DECRETO-LEI -> Decreto-Lei).
    """
    if not isinstance(texto_entrada, str):
        return texto_entrada

    # 1. Remove acentos (Normalização NFKD)
    forma_nfkd = unicodedata.normalize('NFKD', texto_entrada)
    texto = "".join([c for c in forma_nfkd if not unicodedata.combining(c)])

    # 2. Converte para Title Case
    # O Python tratará letras após o hífen como início de nova palavra
    texto = texto.title()

    # 3. Limpeza de caracteres especiais
    # Mantemos letras (a-z, A-Z), números (0-9), hifens (-) e espaços ( )
    # Removemos o restante (como º, ª, parênteses, etc.)
    texto = re.sub(r'[^a-zA-Z0-9\-\s]', '', texto)

    # 4. Ajuste de espaços (opcional)
    # Se quiser transformar espaços em underline mas MANTER o hífen:
    texto = texto.replace(' ', '_')
    
    return texto.strip()

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 24, Finished, Available, Finished, False)

In [23]:
# Crie uma função para limpar sinais gráficos e caracteres especiais
# deixando todas as palavras em maiúsculo

def limpar_string_upper(texto_entrada):
    """
    Higieniza a string e converte todos os caracteres para MAIÚSCULAS.
    Ideal para padronização rigorosa de nomes de arquivos e IDs.
    """
    if not isinstance(texto_entrada, str):
        return texto_entrada

    # 1. Remove acentos (Normalização NFKD)
    forma_nfkd = unicodedata.normalize('NFKD', texto_entrada)
    texto = "".join([c for c in forma_nfkd if not unicodedata.combining(c)])

    # 2. Converte todo o texto para MAIÚSCULAS
    texto = texto.upper()

    # 3. Substitui espaços e hifens por sublinhados
    texto = re.sub(r'[\s\-]+', '_', texto)

    # 4. Remove caracteres especiais (mantém A-Z, 0-9 e _)
    # Note que aqui não precisamos de 'a-z' pois tudo já é maiúsculo
    texto = re.sub(r'[^A-Z0-9_]', '', texto)

    # Remove sublinhados extras nas extremidades
    return texto.strip('_')

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 25, Finished, Available, Finished, False)

#### 1.2.2. Funções específicas para RAG

In [24]:
# Crie funções "helpers" para checar existência de arquivos (Fabric)
def _get_mssparkutils():
    try:
        from notebookutils import mssparkutils
        return mssparkutils
    except Exception:
        try:
            import mssparkutils  # fallback
            return mssparkutils
        except Exception:
            return None

def _fabric_exists(path: str) -> tuple[bool | None, str]:
    """
    Retorna:
      (True,  "ok")  -> existe no Fabric
      (False, "ok")  -> não existe no Fabric (checagem funcionou)
      (None,  "motivo") -> não foi possível checar (sem mssparkutils ou erro)
    """
    ms = _get_mssparkutils()
    if ms is None:
        return None, "mssparkutils_indisponivel"

    try:
        # Checagem por listing do diretório (mais portátil)
        d = os.path.dirname(path) or "."
        f = os.path.basename(path)
        items = ms.fs.ls(d)

        names = set()
        for it in items:
            p = getattr(it, "path", None) or getattr(it, "name", None) or str(it)
            names.add(os.path.basename(str(p)).rstrip("/"))

        return (f in names), "ls_ok"
    except Exception as e:
        return None, f"ls_erro:{repr(e)}"

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 26, Finished, Available, Finished, False)

In [25]:
# Crie funções para gerar arquivos TXT a partir do site do Planalto
# sem dispositivos revogados e preparados para o Azure AI Foundry

# ============================================================
# REGEX CANÔNICO PARA ARTIGOS (aceita milhares "1.000")
# ============================================================

# 10 | 999 | 1.000 | 12.345 | 1.234.567
ART_NUM_RE = r"(?:\d{1,3}(?:\.\d{3})+|\d+)"

def _art_num_to_int(s: str) -> int:
    """Converte '1.000' -> 1000 (apenas para métricas)."""
    return int(s.replace(".", "")) 

# ============================================================
# 1) Normalização final (texto pronto para cracking/chunking)
# ============================================================
def normalize_txt_for_azure_ai_foundry(texto: str) -> str:
    """
    Normaliza texto jurídico extraído do Planalto para consumo direto
    pelo pipeline padrão do Azure AI Foundry (cracking/chunking/embedding).

    Regras:
      (1) Unicode NFC (acentos/símbolos consistentes)
      (2) Corrige padrões típicos do HTML do Planalto (ex.: "§ 1 o", "n º")
      (3) Remove quebras simples dentro do parágrafo, preservando parágrafos
          separados por linha em branco (\n\n)
      (4) Limpeza final de espaços e excesso de linhas em branco
    """     
    
    # (1) Unicode NFC
    texto = unicodedata.normalize("NFC", texto)

    # (2) Correções clássicas do Planalto
    texto = re.sub(r"§\s*(\d+)\s*\n\s*o\b", r"§ \1o", texto)   # § 9\n o -> § 9o
    texto = re.sub(r"\bn\s*\n\s*º\b", r"nº", texto)           # n\nº -> nº
    texto = re.sub(r"\(\s*([^\n()]+)\n\s*([^\n()]+)\)", r"(\1 \2)", texto)  

    # (2.1) Correção clássica adicional: "Art. 1 o" -> "Art. 1º"
    texto = re.sub(
        r"(?im)\bArt\.\s*(\d+)\s*(?:\n\s*)?o\b",
        r"Art. \1º",
        texto
    ) 

    # (3) Junta quebras simples dentro do parágrafo; mantém \n\n
    #     Preserva APENAS estruturas que devem ficar em linha própria:
    #     - incisos "I -", "II -", ...
    #     - parágrafos (§, Parágrafo único)
    #     - cabeçalhos de artigos ("Art. 1º", "Art. 10 o", etc.) - EXCLUIR SE DER CERTO
    #     - cabeçalhos REAIS de artigos (NÃO referências do tipo ‘art. 178 desta Lei’)”
    texto = re.sub(
        r"(?<!\n)\n(?!\n|\s*(?:"
        r"[IVXLCDM]{1,5}\s+-\s+|"     # I - / II - / ...
        r"§\s*\d+º?\s+(?!do\b|da\b|dos\b|das\b|no\b|na\b|nos\b|nas\b|ao\b|aos\b|à\b|às\b)[A-ZÀ-Ü]|" # § 1º, § 2º ...
        r"Par[aá]grafo\b|"                # Parágrafo único                       
        rf"Art\.\s*{ART_NUM_RE}"                        # ✅ Art. 1º, Art. 10 o, Art. 64-A        
        r"(?:\s*[ºo°])?(?:-[A-Z])?(?:\.)?\s+"              # aceita º/o/°, -A e ponto
        r"(?!,?\s*(?:da|do|das|dos|no|na|nos|nas|ao|aos|à|às)\b)"  # anti-referência
        r"[A-ZÀ-Ü]"                                       # próximo caractere MAIÚSCULO => cara de cabeçalho      
        r"))",
        " ",
        texto,
        flags=re.I
    )   
    
    # (4) Limpeza final
    texto = re.sub(r"[ \t]{2,}", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    return texto.strip()

# ============================================================
# 2) Fetch robusto com fallback
# ============================================================
def _looks_like_js_block(html: str) -> bool:
    """Detecta páginas que dependem de JS (tendem a ser inúteis sem browser)."""
    if not html:
        return True
    h = html.lower()
    return ("please enable javascript" in h) or ("your support id" in h)

def _fetch_html_with_fallback(
    session: requests.Session,
    url: str,
    headers: dict,
    timeout: tuple[int, int],
    sanity_regex: str = rf"\bArt\.\s*{ART_NUM_RE}\b"
) -> tuple[str, str]:
    """
    Retorna (html, final_url_used).

    Considera válido apenas HTML que:
      - contenha padrão normativo mínimo (ex.: 'Art. 1º')
      - não seja página que exige JavaScript

    Fallbacks suportados:
      (1) URL original
      (2) http:// (se original era https://)
    """

    def decode(resp: requests.Response) -> str:
        enc = resp.apparent_encoding or "utf-8"
        return resp.content.decode(enc, errors="replace")

    def ok(html: str) -> bool:
        return bool(re.search(sanity_regex, html)) and not _looks_like_js_block(html)

    tried = []

    # (1) original
    r1 = session.get(url, headers=headers, timeout=timeout)
    html1 = decode(r1)
    tried.append(r1.url)
    if ok(html1):
        return html1, r1.url

    # (2) http fallback
    if url.lower().startswith("https://"):
        url_http = "http://" + url.split("://", 1)[1]
        r2 = session.get(url_http, headers=headers, timeout=timeout)
        html2 = decode(r2)
        tried.append(r2.url)
        if ok(html2):
            return html2, r2.url

    # Falha controlada: retorna HTML original + trilha
    return html1, " | ".join(tried)

# ============================================================
# LOG XLSX (aba LOG + Glossário)
# ============================================================
def _append_log_xlsx(log_xlsx_path: str, row: dict) -> None:
    """
    Acrescenta uma linha a um LOG em formato XLSX.
    - Aba principal: 'LOG' (substitui Sheet1)
    - Aba auxiliar: 'Glossário' (descrição das métricas)
    - Ordenação do LOG: legislacao_planalto_tipo (crescente)        
    """

    # Garante que o diretório exista 
    os.makedirs(os.path.dirname(log_xlsx_path) or ".", exist_ok=True)

    # ----------------------------
    # 1) LOG (aba principal)
    # ----------------------------

    # 1.1) Nova linha: converte o dict da execução em DataFrame (1 linha)
    df_new = pd.DataFrame([row])

    # 1.2) Lê LOG existente (se houver)
    if os.path.exists(log_xlsx_path):
        try:
            df_existing = pd.read_excel(log_xlsx_path, sheet_name="LOG", engine="openpyxl")
        except Exception:
            # fallback: caso exista mas ainda esteja como Sheet1
            df_existing = pd.read_excel(log_xlsx_path, sheet_name=0, engine="openpyxl")
        
        # Concatena preservando colunas
        df_log = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        # Primeiro registro do log
        df_log = df_new   
    
    # ----------------------------
    # 2) Glossário (aba auxiliar)
    # ----------------------------

    # Glossário — duas colunas: Coluna_LOG / Mede ou informa o quê
    df_glossario = pd.DataFrame(
        [
            ("html_raw_chars", "Quantidade de caracteres do HTML bruto retornado pelo site do Planalto"),
            ("visible_text_chars", "Quantidade de caracteres do texto visível após remoção de HTML, scripts e estilos"),
            ("txt_vigente_chars", "Quantidade de caracteres do texto final vigente salvo em TXT"),
            ("txt_revogado_chars", "Quantidade de caracteres do texto consolidado com trechos revogados"),
            ("removed_strike_count", "Número de blocos HTML visualmente tachados (<del>, <s>, <strike>, line-through) removidos"),
            ("removed_revogado_lines_count", "Número de linhas do texto contendo explicitamente a marca '(Revogado)'"),
            ("last_art_expected", "Maior número de artigo identificado no HTML completo do normativo (referência esperada)"),
            ("last_art_extracted", "Maior número de artigo identificado no texto vigente extraído (resultado efetivo da extração)"),
        ],
        columns=["Coluna_LOG", "Mede ou informa o quê"]
    )
    
    # ----------------------------
    # 3) Critério_Alerta (aba auxiliar)       
    # ----------------------------
    
    df_criterio_alerta = pd.DataFrame(
        [
            (
                1,
                "HTML muito grande e texto muito pequeno (caso típico de extração incompleta)",
                "HTML_RAW_CHARS >= 200000 AND VISIBLE_TEXT_CHARS <= 5000",
                "ALERTA_HTML_GRANDE_TEXTO_PEQUENO",
            ),
            (
                2,
                "Texto vigente muito curto (limiar conservador)",
                "TXT_VIGENTE_CHARS < 2000",
                "ALERTA_TEXTO_CURTO",
            ),
            (
                3,
                "Razão texto/HTML muito baixa",
                "VISIBLE_TEXT_CHARS / HTML_RAW_CHARS < 0.2 (COM HTML_RAW_CHARS > 0)",
                "ALERTA_EXTRACAO_INCOMPLETA",
            ),
            (
                4,
                "Muitos '(Revogado)' com texto total pequeno (reforço)",
                "REMOVED_REVOGADO_LINES_COUNT >= 10 AND VISIBLE_TEXT_CHARS <= 5000",
                "ALERTA_EXTRACAO_INCOMPLETA",
            ),            
            (
                5,
                "Último artigo extraído menor que o detectado no HTML",
                "LAST_ART_EXTRACTED < LAST_ART_EXPECTED",
                "ALERTA_TERMINOU_CEDO",
            ),

        ],
        columns=[
            "REGRA_NUM",
            "REGRA_DESCRICAO",
            "ALERTA_CRITERIO",
            "ALERTA_DESCRICAO",
        ],
    )

    # ----------------------------
    # 4) Escrita final do XLSX
    # ----------------------------
    
    # Escreve XLSX com duas abas (LOG + Glossário)
    # Salva (sobrescreve o XLSX de forma controlada)
    with pd.ExcelWriter(
        log_xlsx_path,
        engine="openpyxl",
        mode="w"
    ) as writer:
        df_log.to_excel(writer, sheet_name="LOG", index=False)
        df_glossario.to_excel(writer, sheet_name="Glossário", index=False)
        df_criterio_alerta.to_excel(writer, sheet_name="Critério_Alerta", index=False)

# ============================================================
# 4) Função principal (para ser aplicada em várias URLs do df_planalto)
# ============================================================
def html_planalto_to_txt(
    url: str,
    pasta_secundaria_fabric: str | None = None,
    # --- LEGISLAÇÃO VIGENTE ---
    arquivo_vigente_nome: str | None = None,       # ex.: df_planalto["ARQUIVO"] (pode vir com .txt)
    output_vigente_dir: str = ".",
    base_vigente_filename: str | None = None,
    
    # --- LEGISLAÇÃO APENAS COM TRECHOS REVOGADOS ---
    output_revogado: bool = False,                 # se True, gera e (por padrão) salva o revogado
    output_revogado_dir: str | None = None,         # se None, usa output_vigente_dir
    arquivo_revogado_nome: str | None = None,       # opcional: nome específico do revogado (pode vir com .txt)
    base_revogado_filename: str | None = None,      # opcional: base específica do revogado

    # --- LOG (xlsx) ---
    log_xlsx_path: str | None = None,

    timeout: tuple[int, int] = (10, 90),
    headers: dict | None = None,
) -> dict:
    """
    Baixa HTML do Planalto, separa texto vigente vs. revogado e exporta TXT (UTF-8)
    pronto para Azure AI Foundry.

    Nome dos arquivos (vigente e revogado):
      Prioridade para definir a BASE:
        (1) arquivo_*_nome (se vier com ".txt", remove a extensão)
        (2) base_*_filename
        (3) inferido da URL

    Saídas:
      - {base_vigente}_vigente.txt  (sempre)
      - {base_revogado}_revogado.txt (somente se output_revogado=True)

    Observações:
      - txt_revogado_chars é SEMPRE calculado (mesmo sem salvar).
      - Quando output_revogado=False, o arquivo *_revogado.txt NÃO é salvo.
      - output_revogado_dir permite salvar revogados em caminho separado (staging ou definitivo).
    """

    # (0) Headers padrão
    if headers is None:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0 Safari/537.36"
            )
        }

    # (1) Session + retry
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    # (2) Fetch robusto
    html, final_url_used = _fetch_html_with_fallback(
        session=session,
        url=url,
        headers=headers,
        timeout=timeout,        
        sanity_regex = rf"\bArt\.\s*{ART_NUM_RE}(?:\s*[ºo°])?",        
    )
    html_raw_chars = len(html)

    soup = BeautifulSoup(html, "html.parser")

    # (3) Remover lixo
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # (4) Remover e guardar tachados por tag
    revogados_parts = []
    removed_strike_count = 0

    for tag in soup.find_all(["del", "s", "strike"]):
        txt = tag.get_text(" ", strip=True)
        if txt:
            revogados_parts.append(txt)
        # Contagem de trechos com '<del>...</del>', '<s>...</s>' e '<strike>...</strike>'
        # (ou seja, mede quantos trechos HTML visualmente tachados (revogados) existiam
        # na página e foram removidos do texto vigente durante o processamento)        
        removed_strike_count += 1
        tag.decompose()       

    # (5) Remover e guardar tachados por style (robusto para HTML legado)
    for tag in list(soup.find_all(True, attrs={"style": True})):

        # 5.1) Garante que é Tag de verdade
        if not isinstance(tag, Tag):
            continue

        # 5.2) Garante que attrs existe e é dict (HTML legado pode corromper isso)
        attrs = getattr(tag, "attrs", None)
        if not isinstance(attrs, dict):
            continue

        # 5.3) Lê style com segurança
        style = attrs.get("style")
        if not isinstance(style, str):
            continue

        # 5.4) Aplica regra do tachado
        if re.search(r"line-through", style, flags=re.I):
            txt = tag.get_text(" ", strip=True)
            if txt:
                revogados_parts.append(txt)
            removed_strike_count += 1
            tag.decompose()
    
    # (6) Preservar <br>
    for br in soup.find_all("br"):
        br.replace_with("\n")

    # (7) Extrair texto em blocos
    container = soup.body if soup.body else soup
    block_tags = {
        "p", "div", "li", "tr", "td", "blockquote", "center",
        "h1", "h2", "h3", "h4", "h5", "h6"
    }

    paragraphs = []
    seen = set()

    for el in container.find_all(list(block_tags)):
        raw = el.get_text(separator="\n")
        raw = "\n".join(re.sub(r"[ \t]+", " ", l).strip() for l in raw.splitlines())
        raw = "\n".join(l for l in raw.splitlines() if l)
        if not raw:
            continue

        key = re.sub(r"\s+", " ", raw)
        if key in seen:
            continue
        seen.add(key)

        paragraphs.append(raw)

    visible_text = "\n\n".join(paragraphs).strip()
    visible_text_chars = len(visible_text)    

    # --- Fallback baseado em CABEÇALHOS de artigos (não em referências) ---    
    def _max_art_heading_num(texto: str) -> int | None:
        """
        Maior número de artigo considerando APENAS cabeçalhos reais:
        - início de linha
        - 'Art.' + número
        - aceita 'º' e '.' após o número (ex.: 'Art. 10.' / 'Art. 9º')
        - aceita sufixo -A/-B (captura só a parte numérica)
        - depois disso, exige que a próxima palavra NÃO seja preposição minúscula (da/do/no/na...)
        - e que o primeiro caractere do conteúdo seja MAIÚSCULO.
        """
        if not texto:
            return None

        pattern = re.compile(
            rf"(?im)^\s*Art\.\s*({ART_NUM_RE})"          # Art. 64, Art. 1.000
            r"\s*(?:[ºo°])?"                  # aceita º OU o OU °, com/sem espaço
            r"\s*(?:-[A-Z])?"                 # opcional: -A, -B...
            r"\s*(?:\.)?"                     # opcional: ponto após o número
            r"\s+"                            # espaço obrigatório
            r"(?!da\b|do\b|das\b|dos\b|no\b|na\b|nos\b|nas\b|ao\b|aos\b|à\b|às\b)"
            r"([A-ZÀ-Ü])"                     # próximo caractere deve ser MAIÚSCULO
        )
              
        nums = []
        for m in pattern.finditer(texto):
            try:
                nums.append(_art_num_to_int(m.group(1)))  # ✅ "1.000" -> 1000
            except Exception:
                pass

        return max(nums) if nums else None


    # Texto completo do HTML (pega inclusive o que ficou fora do <body>)
    full_text_all = soup.get_text(separator="\n", strip=True)
    full_text_all_norm = re.sub(r"[ \t]+", " ", full_text_all)
    full_text_all_norm = re.sub(r"\n{3,}", "\n\n", full_text_all_norm).strip()

    # Texto obtido pelo método atual (por blocos)
    visible_text_norm = re.sub(r"[ \t]+", " ", visible_text)
    visible_text_norm = re.sub(r"\n{3,}", "\n\n", visible_text_norm).strip()

    # Compara CABEÇALHOS  
    last_head_all = _max_art_heading_num(full_text_all_norm) or 0
    last_head_blocks = _max_art_heading_num(visible_text_norm) or 0

    # DEBUG útil (vai te dar exatamente o diagnóstico correto)
    # print(f"DEBUG: last_head_all = {last_head_all} | last_head_blocks = {last_head_blocks}")

    if last_head_all > last_head_blocks:
        visible_text = full_text_all_norm
    else:
        visible_text = visible_text_norm

    visible_text_chars = len(visible_text)
           
    
    # DEBUG FINAL: confirmação de artigos após fallback    
    # print(
        # "DEBUG FINAL: último Art. (cabeçalho) no visible_text =",
        # _max_art_heading_num(visible_text)
    # )    
    
    # (8) Separar linhas "(Revogado)"
    removed_revogado_lines = []
    vigente_lines = []
    removed_revogado_lines_count = 0

    for line in visible_text.splitlines():
        if re.search(r"\(\s*revogado\s*\)", line, flags=re.I):
            removed_revogado_lines.append(line.strip())
            removed_revogado_lines_count += 1
        else:
            vigente_lines.append(line)

     
    # (9) Normalizar texto vigente 
    # ---------- Helpers internos para estruturar parágrafos ----------               
    def _ensure_foundry_paragraphs(text: str) -> str:
        """
        Reforça parágrafos estruturais do texto normativo do Planalto.
        Otimizada para cracking / chunking / embedding no Azure AI Foundry.

        Blindagens importantes:
        - NÃO cria parágrafo para referências do tipo:
        "art. 173 da Constituição", "art. 216, da Constituição", etc.
        - Artigos, parágrafos e incisos ficam em blocos separados.
        """
        if not text:
            return text

        # 0) Normaliza NBSP (muito comum no Planalto)
        text = text.replace("\xa0", " ")
        text = re.sub(r"&(?:amp;)?nbsp;+", " ", text, flags=re.I)

        # 0.1) ÍNDICE e PARTES grafadas com letras espaçadas (estrutura pré-normativa)
        # Força quebra de parágrafo ANTES desses cabeçalhos
        text = re.sub(
            r"(?im)(^|[^\n])\s*(ÍNDICE)\b",
            r"\1\n\n\2",
            text
        )

        # P A R T E    G E R A L
        text = re.sub(
            r"(?im)(^|[^\n])\s*(P\s+A\s+R\s+T\s+E\s+G\s+E\s+R\s+A\s+L)\b",
            r"\1\n\n\2",
            text
        )

        # P A R T E    E S P E C I A L
        text = re.sub(
            r"(?im)(^|[^\n])\s*(P\s+A\s+R\s+T\s+E\s+E\s+S\s+P\s+E\s+C\s+I\s+A\s+L)\b",
            r"\1\n\n\2",
            text
        )
        
        # 1) PARTE / LIVRO / TÍTULO / CAPÍTULO / SEÇÃO (somente com numeral romano ou palavra-chave)
        text = re.sub(
            r"(?im)(^|[.:;])\s*("
            r"(?:PARTE\s+[A-ZÀ-Ü]+|"            # PARTE GERAL / PARTE ESPECIAL
            r"LIVRO\s+[IVXLCDM]+|"               # LIVRO I
            r"T[IÍ]TULO\s+[IVXLCDM]+|"           # TÍTULO I
            r"CAP[IÍ]TULO\s+[IVXLCDM]+|"         # CAPÍTULO I
            r"SE[CÇ][AÃ]O\s+[IVXLCDM]+)"          # SEÇÃO I
            r"(?:\s+[A-ZÀ-Ü][^\n]*)?"
            r")\b",
            r"\1\n\n\2",
            text
        )

        # 2) ARTIGOS — somente cabeçalhos reais (blindado contra referências)
        text = re.sub(
            r"(?im)(^|[.:;])\s*"
            rf"(Art\.\s*{ART_NUM_RE}(?:\s*[ºo°])?(?:-[A-Z])?(?:\.)?\s+"
            r"(?!,?\s*(?:da|do|das|dos|no|na|nos|nas|ao|aos|à|às)\b)"
            r"[A-ZÀ-Ü])",
            r"\1\n\n\2",
            text
        )
        
        # 2.1) Se já está no começo da linha mas só com \n simples antes, promove para \n\n
        text = re.sub(
            rf"(?m)(?<!\n)\n(?=\s*Art\.\s*{ART_NUM_RE}\b)",
            "\n\n",
            text
        )

        # 3) Parágrafo único
        text = re.sub(
            r"(?im)(^|[.:;])\s*(Par[aá]grafo\s+único\.)",
            r"\1\n\n\2",
            text
        )         

        # 4) §§ — cada § inicia parágrafo APENAS quando estrutural
        text = re.sub(
            r"(?m)(^|[.:;])\s*(§\s*\d+º?\s+"
            r"(?!do\b|da\b|dos\b|das\b|no\b|na\b|nos\b|nas\b|ao\b|aos\b|à\b|às\b))",
            r"\1\n\n\2",
            text
        )

        # 5) INCISOS — cada inciso em bloco próprio (linha em branco)
        text = re.sub(
            r"(?m)^\s*([IVXLCDM]{1,5}\s+-\s+)",
            r"\n\n\1",
            text
        )

        # 6) Limpa excesso de linhas em branco
        text = re.sub(r"\n{3,}", "\n\n", text).strip()

        return text

    # Texto vigente bruto
    texto_vigente = re.sub(r"\n{3,}", "\n\n", "\n".join(vigente_lines)).strip()

    # 1) Estrutura títulos e artigos (robusto para ALL CAPS e Title Case)
    #    Reforça parágrafos estruturais para Foundry antes da normalização
    texto_vigente = _ensure_foundry_paragraphs(texto_vigente)    

    # 1.5) Normaliza finais de linha (blindagem contra \r\n / \r)
    texto_vigente = texto_vigente.replace("\r\n", "\n").replace("\r", "\n")   
    
    # 2) Remove parágrafo indevido dentro de frase
    #    NÃO colapsa antes de "Art. <n>" (estrutura normativa)
    texto_vigente = re.sub(
        r"\n\s*\n\s*(?=(?:inciso\b|al[ií]nea\b|caput\b|e\b|ou\b|no\b|na\b|do\b|da\b))",
        " ",
        texto_vigente,
        flags=re.I
    )
    
    # 2.1) Corrige quebra de PARÁGRAFO indevida após preposição e antes de
    # referências normativas INLINE (art., Capítulo, Seção, Título, Livro)
    NBSP_RE = r"[\s\u00A0\u202F\u2007\u2060]*"

    texto_vigente = re.sub(
        rf"(?im)"
        rf"(\b(?:no|na|nos|nas|do|da|dos|das|ao|aos|à|às|em|num|numa|nuns|numas)\b)"
        rf"{NBSP_RE}(?:\r?\n{NBSP_RE}){{2,}}"
        rf"(?=("
        rf"(?:art|arts)\.\s*{ART_NUM_RE}(?:\s*[ºo°])?(?!\d)"
        rf"|Cap[ií]tulo\s+[IVXLCDM]+"
        rf"|Se[cç][aã]o\s+[IVXLCDM]+"
        rf"|T[ií]tulo\s+[IVXLCDM]+"
        rf"|Livro\s+[IVXLCDM]+"
        rf")\b)",
        r"\1 ",
        texto_vigente
    )

    # ALTERADO AQUI (INÍCIO)
    # 2.2) Força quebra estrutural ANTES do fecho formal do ato:
    # "Brasília, <dia> de <mês> de <ano>; <n><º/o/°> da Independência e <n><º/o/°> da República."
    #
    # Robustez:
    # - aceita "181º", "181o" e "181 o"
    # - aceita "114º", "114o" e "114 o"
    # - funciona mesmo se "Brasília," estiver no meio da linha após um ponto final

    MES_RE = r"[A-Za-zÀ-ÿ]+"
    ORD_RE = r"\d+\s*(?:º|o|°)"  # <- aqui está a diferença crítica p/ o Código Civil

    texto_vigente = re.sub(
        rf"(?im)"
        rf"([.!?])\s+(Bras[íi]lia,\s*\d{{1,2}}\s+de\s+{MES_RE}\s+de\s+\d{{4}};"
        rf"\s*{ORD_RE}\s+da\s+Independência\s+e\s+{ORD_RE}\s+da\s+República\.)",
        r"\1\n\n\2",
        texto_vigente
    )
    # ALTERADO AQUI (FIM)
    
    # 3) Força parágrafo antes de § APENAS quando for estrutural
    #    (não quebra referências como "§ 2º do art. 216 da Constituição")
    texto_vigente = re.sub(
        r"(?m)^\s*(§\s*\d+º?\s+"
        r"(?!do\b|da\b|dos\b|das\b|no\b|na\b|nos\b|nas\b))",
        r"\n\n\1",
        texto_vigente
    ) 
     
    # 4) Quebra incisos em NOVO PARÁGRAFO, preservando ":" se existir
    texto_vigente = re.sub(
        r"(?<!\n)(:\s*)([IVXLCDM]{1,5}\s+-\s+)",
        r"\1\n\n\2",
        texto_vigente
    )

    texto_vigente = re.sub(
        r"(?<!\n)\s+([IVXLCDM]{1,5}\s+-\s+)",
        r"\n\n\1",
        texto_vigente
    )   

    # Defesa final: cada inciso começa em parágrafo próprio
    texto_vigente = re.sub(
        r"(?m)^\s*([IVXLCDM]{1,5}\s+-\s+)",
        r"\n\n\1",
        texto_vigente
    )  

    # 5) Normalização final
    texto_final_vigente = normalize_txt_for_azure_ai_foundry(texto_vigente)

    # 6) last_art_extracted (depois da normalização) + fallback
    # ---------- Helpers internos para identificar último artigo ----------
    def _max_art_heading_num_fallback(texto: str) -> int | None:
        """
        Fallback mais permissivo, mas ainda restrito a cabeçalhos:
        - começo de linha (^)
        - aceita ordinal º/o/° com ou sem espaço
        - rejeita referências típicas: "Art. 173 da/do/das..." etc.
        - não exige que o próximo caractere seja maiúsculo (permite ALL CAPS/Title Case variados)
        """
        if not texto:
            return None

        pattern = re.compile(
            rf"(?im)^\s*Art\.\s*({ART_NUM_RE})"  # Art. 64, Art. 1.000
            r"\s*(?:[ºo°])?"                  # º ou o ou °
            r"\s*(?:-[A-Z])?"                 # -A, -B
            r"\s*(?:\.)?"                     # ponto opcional
            r"\s+"                            # precisa ter “texto” depois
            r"(?!,?\s*(?:da|do|das|dos|no|na|nos|nas|ao|aos|à|às)\b)"  # anti-referência
        )       
        
        nums = []
        for m in pattern.finditer(texto):
            try:
                nums.append(_art_num_to_int(m.group(1)))
            except Exception:
                pass
        return max(nums) if nums else None   

    # ✅ last_art_extracted = cabeçalho real
    last_art_extracted = (
        _max_art_heading_num(texto_final_vigente)
        or _max_art_heading_num_fallback(texto_final_vigente)
        or 0
    )       
      
    # (9.1) Métrica do TXT vigente (sempre definida, mesmo se falhar ao salvar)
    txt_vigente_chars = len(texto_final_vigente)

    # (10) Texto revogado consolidado
    texto_revogado = "\n\n---\n\n".join(
        [t for t in revogados_parts if t] +
        [l for l in removed_revogado_lines if l]
    ).strip()

    texto_revogado = normalize_txt_for_azure_ai_foundry(texto_revogado)

    # ---------- Helpers internos para definir base ----------
    def _derive_base(nome_arquivo: str | None, base_filename: str | None) -> str:
        if nome_arquivo:
            b = os.path.basename(str(nome_arquivo).strip())
            b = re.sub(r"\.txt$", "", b, flags=re.I)  # remove .txt/.TXT
        elif base_filename:
            b = str(base_filename).strip()
            b = re.sub(r"\.txt$", "", b, flags=re.I)
        else:
            path_last = urlparse(url).path.rstrip("/").split("/")[-1] or "documento"
            b = path_last

        # sanitização segura para Fabric/Lakehouse
        b = re.sub(r"[^A-Za-z0-9_.-]+", "_", b).strip("_") or "documento"
        return b

    # (11) Definir BASE do vigente
    base_vigente = _derive_base(arquivo_vigente_nome, base_vigente_filename)

    # (11b) Definir BASE do revogado (por padrão, usa a mesma do vigente)
    # Se você não passar arquivo_revogado_nome/base_revogado_filename, a base fica igual.
    if arquivo_revogado_nome or base_revogado_filename:
        base_revogado = _derive_base(arquivo_revogado_nome, base_revogado_filename)
    else:
        base_revogado = base_vigente

    # (12) Salvar TXT vigente (tenta salvar; se falhar, marca ERRO_IO)   
    arquivo_vigente_gerado = "NAO"
    arquivo_vigente_modo = "NAO_SALVO" 
    status_execucao = "NAO_SALVO_ERRO_OUTRO"
    erro_execucao = ""

    try:
        os.makedirs(output_vigente_dir, exist_ok=True)
        out_vigente = os.path.join(output_vigente_dir, f"{base_vigente}_vigente.txt")

        # 12.0) Detecta se o arquivo já existia ANTES
        existed_before = os.path.exists(out_vigente)

        # 12.1) Escreve (modo "w" sobrescreve se existir)
        with open(out_vigente, "w", encoding="utf-8") as f:
            f.write(texto_final_vigente)

        # 12.2) Verificação pós-escrita (evita log "SIM" sem arquivo)
        exists_after = os.path.isfile(out_vigente)
        size_after = os.path.getsize(out_vigente) if exists_after else 0

        if exists_after and size_after > 0:
            arquivo_vigente_gerado = "SIM"
            arquivo_vigente_modo = "SOBRESCRITO" if existed_before else "CRIADO"
            status_execucao = "SALVO"
        else:
            # escrita “aparentemente” ocorreu, mas arquivo não ficou acessível/persistido
            arquivo_vigente_gerado = "NAO"
            arquivo_vigente_modo = "NAO_SALVO"
            status_execucao = "NAO_PERSISTIDO"
            erro_execucao = "write_ok_mas_arquivo_nao_encontrado_ou_tamanho_zero_pos_write"

    except OSError as e:
        # erro de escrita (permissão, path, quota, etc.)
        arquivo_vigente_gerado = "NAO"
        arquivo_vigente_modo = "NAO_SALVO"
        status_execucao = "NAO_SALVO_ERRO_IO"
        erro_execucao = repr(e)

    except Exception as e:
        arquivo_vigente_gerado = "NAO"
        arquivo_vigente_modo = "NAO_SALVO"
        status_execucao = "NAO_SALVO_ERRO_OUTRO"
        erro_execucao = repr(e)     
     
    # (13) Critérios de alerta

    # Métrica de revogado (sempre)
    txt_revogado_chars = len(texto_revogado)

    # status_qualidade (alertas derivados das métricas)
    # Observação: heurísticas de alerta (não substituem validação humana).
    alertas = []

    # Regra 1: HTML muito grande e texto muito pequeno (caso típico de extração incompleta)
    if (html_raw_chars is not None) and (visible_text_chars is not None):
        if html_raw_chars >= 200_000 and visible_text_chars <= 5_000:
            alertas.append("ALERTA_HTML_GRANDE_TEXTO_PEQUENO")

    # Regra 2: texto vigente muito curto (limiar conservador)
    if txt_vigente_chars is not None and txt_vigente_chars < 2_000:
        alertas.append("ALERTA_TEXTO_CURTO")

    # Regra 3: razão texto/HTML muito baixa (guarda-chuva)
    if (html_raw_chars is not None) and (html_raw_chars > 0) and (visible_text_chars is not None):
        ratio = visible_text_chars / html_raw_chars
        if ratio < 0.2:
            alertas.append("ALERTA_EXTRACAO_INCOMPLETA")

    # Regra 4: muitos "(Revogado)" com texto total pequeno (reforço)
    if (removed_revogado_lines_count is not None) and (visible_text_chars is not None):
        if removed_revogado_lines_count >= 10 and visible_text_chars <= 5_000:
            alertas.append("ALERTA_EXTRACAO_INCOMPLETA")    
      
    # Regra 5: último artigo extraído < último artigo detectado no HTML completo)
    last_art_expected = _max_art_heading_num(full_text_all_norm)   
    if (last_art_expected is not None) and (last_art_extracted is not None):
        if last_art_extracted < last_art_expected:
            alertas.append("ALERTA_TERMINOU_CEDO")
    
    status_qualidade = "SEM_ALERTA" if not alertas else "|".join(sorted(set(alertas)))

    # (14) Caminho e salvamento do TXT revogado (mesma lógica do vigente, mas com dir próprio)
    output_revogado_path = None
    if output_revogado:
        # se não vier dir de revogado, usa o do vigente
        rev_dir = output_revogado_dir if output_revogado_dir is not None else output_vigente_dir
        os.makedirs(rev_dir, exist_ok=True)

        output_revogado_path = os.path.join(rev_dir, f"{base_revogado}_revogado.txt")
        with open(output_revogado_path, "w", encoding="utf-8") as f:
            f.write(texto_revogado)    

    # (15) LOG XLSX — horário de Brasília (ISO‑8601)   
    result = {
        "legislacao_planalto_tipo": pasta_secundaria_fabric,

        # "base_vigente": base_vigente,
        "arquivo_vigente_nome_original": arquivo_vigente_nome,
        # "arquivo_vigente_gerado": arquivo_vigente_gerado,
        "arquivo_vigente_modo": arquivo_vigente_modo,
        # "status_execucao": status_execucao,
        "status_qualidade": status_qualidade,       
        # "output_vigente_dir": output_vigente_dir,
        # "output_vigente_path": out_vigente,             

        # "base_revogado": base_revogado,
        # "arquivo_revogado_nome_original": arquivo_revogado_nome,
        # "output_revogado_dir": (output_revogado_dir if output_revogado_dir is not None else output_vigente_dir) if output_revogado else None,
        # "output_revogado_path": output_revogado_path,

        "html_raw_chars": html_raw_chars,
        "visible_text_chars": visible_text_chars,
        "txt_vigente_chars": txt_vigente_chars,
        "txt_revogado_chars": txt_revogado_chars,
        "removed_strike_count": removed_strike_count,
        "removed_revogado_lines_count": removed_revogado_lines_count, 
        "last_art_expected": last_art_expected,
        "last_art_extracted": last_art_extracted,      
        
        # "foundry_ready": True,
        # "output_revogado_enabled": output_revogado,

        # "source_url": url,
        "final_url_used": final_url_used,
    }   

    # Opcional: registrar o erro (quando houver) para auditoria
    if erro_execucao:
        result["erro_execucao"] = erro_execucao

    if log_xlsx_path:
        log_row = {
            **result,
            "timestamp_iso": datetime.now(ZoneInfo("America/Sao_Paulo")).isoformat(timespec="seconds"),
        }
        _append_log_xlsx(log_xlsx_path, log_row)

    return result

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 27, Finished, Available, Finished, False)

### 1.3. Localização do caminho-padrão do Lakehouse no Fabric

In [26]:
# Define o caminho padrão do Lakehouse no Fabric
fabric_path = Path("/lakehouse/default/Files")

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 28, Finished, Available, Finished, False)

In [27]:
# Lógica: Se o diretório do Fabric existir, use-o. 
# Caso contrário, use o diretório de trabalho atual (CWD) - que será o '/content' do Colab.
base_path = fabric_path if fabric_path.exists() else Path.cwd()

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 29, Finished, Available, Finished, False)

### 1.4. Localização do Inventário de Fontes para RAG

In [28]:
# Crie variáveis tanto para o nome do arquivo do Inventário de Fontes para RAG, quanto
# para o nome da pasta onde ele será armazenado no Lakehouse do Fabric
inventario_fontes_arquivo = 'Inventario_Fontes_RAG_Integridade_STN_ASRCC.xlsm'
inventario_fontes_pasta = 'fontes_enderecos_web_dctos'

# Informe o caminho do arquivo no Lakehouse do Fabric
inventario_fontes_path = base_path / inventario_fontes_pasta / inventario_fontes_arquivo

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 30, Finished, Available, Finished, False)

### 1.5. Localização dos arquivos TXT com as legislações vigentes

In [29]:
# Crie variável para o nome da pasta que conterá outras pastas com os arquivos TXT
# utilizados como fontes para criação de índices do RAG
legislacao_planalto_vigente_pasta = 'documentos_para_indice'

# Informe o caminho da pasta no Lakehouse do Fabric
legislacao_planalto_vigente_path = base_path / legislacao_planalto_vigente_pasta

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 31, Finished, Available, Finished, False)

### 1.6. Localização dos arquivos TXT com trechos das legislações revogadas

In [30]:
# Crie variável para o nome da pasta que conterá outras pastas com os arquivos TXT
# gerados com trechos de legislações revogadas
legislacao_planalto_revogada_pasta = 'legislacao_planalto_revogada'

# Informe o caminho da pasta no Lakehouse do Fabric
legislacao_planalto_revogada_path = base_path / legislacao_planalto_revogada_pasta

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 32, Finished, Available, Finished, False)

### 1.7. Localização dos arquivos de log

In [31]:
# Crie variável para o nome da pasta que conterá o arquivo de log
log_pasta = 'log_legislacao_planalto'

# Informe o caminho da pasta no Lakehouse do Fabric
log_path = base_path / log_pasta

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 33, Finished, Available, Finished, False)

## **2. Criação do DataFrame "Inventário Fontes RAG"**

In [32]:
# Crie um DataFrame a partir da aba "Inventario_Fontes_RAG" do
# arquivo de Inventário de Fontes para RAG

# Carregamento da aba do Excel com as primeiras 3 linhas e 2 colunas
df_inventario_fontes = pd.read_excel(
    inventario_fontes_path,
    sheet_name = 'Inventario_Fontes_RAG'     
)

# Limpe caracteres dos nomes das colunas do DataFrame
df_inventario_fontes.columns = [limpar_string_upper(col) if isinstance(col, str) else col for col in df_inventario_fontes.columns]

# Exiba o DataFrame resultante
df_inventario_fontes.head()

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 34, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,ID,NORMA_DESCRICAO,NORMA_TIPO,NORMA_NUM,NORMA_ANO,EMENTA_DESCRICAO,TEMA_PRINCIPAL,PASTA_PRINCIPAL_FABRIC,PASTA_SECUNDARIA_FABRIC,ARQUIVO,URL,EDITAR,ATUALIZACAO,UNID_RESP_CURADORIA,SERVIDOR_RESP_CURADORIA,OBSERVACAO_NUEIA,OBSERVACAO_AREA_NEGOCIO
0,1,Constituição Federal de 1988,Constituição Federal,NaN,1988.0,"Texto constitucional compilado, contendo os pr...",Diversos,Legislação Planalto,Constituição Federal,CONSTITUICAO FEDERAL_1988_COMPILADA.txt,https://www.planalto.gov.br/ccivil_03/constitu...,Não,Código Python,NaN,NaN,NaN,NaN
1,2,Decreto-Lei nº 2.848/1940 (Código Penal),Decreto-Lei,2848.0,1940.0,"Define os crimes e as penas, incluindo crimes ...",Penal,Legislação Planalto,Decreto-Lei,DECRETO-LEI_2848_1940_COMPILADO - CODIGO PENAL...,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
2,3,Decreto-Lei nº 3.689/1941 (Código de Processo ...,Decreto-Lei,3689.0,1941.0,Regula o processo penal em todo o território b...,Penal,Legislação Planalto,Decreto-Lei,DECRETO-LEI_3689_1941_COMPILADO - CODIGO DE PR...,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
3,4,Decreto-Lei nº 4.657/1942 (LINDB),Decreto-Lei,4657.0,1942.0,Lei de Introdução às normas do Direito Brasile...,NaN,Legislação Planalto,Decreto-Lei,DECRETO-LEI_4657_1942_COMPILADO - LINDB.txt,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
4,5,Lei nº 10.406/2002 (Código Civil),Lei Ordinária,10406.0,2002.0,Institui o Código Civil Brasileiro.,NaN,Legislação Planalto,Lei Ordinária,LEI_10406_2002_COMPILADA - CODIGO_CIVIL.txt,https://www.planalto.gov.br/ccivil_03/leis/200...,Não,Código Python,NaN,NaN,NaN,NaN


In [33]:
# Converte para numérico e depois para o tipo Inteiro que aceita nulos (Int64)
for col in ['NORMA_NUM', 'NORMA_ANO']:
    df_inventario_fontes[col] = (
        pd.to_numeric(df_inventario_fontes[col], errors='coerce') # Garante que são números (float)
        .astype('Int64')                                         # Converte para Inteiro e remove o .0
    )

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 35, Finished, Available, Finished, False)

In [34]:
# Edite a coluna "PASTA_PRINCIPAL_FABRIC" para que o seu conteúdo apresente 
# apenas a primeira letra das palavras em maiúsculo 
df_inventario_fontes['PASTA_PRINCIPAL_FABRIC'] = df_inventario_fontes['PASTA_PRINCIPAL_FABRIC'].apply(limpar_string_title)

# Exiba o DataFrame resultante
df_inventario_fontes.head()

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 36, Finished, Available, Finished, False)

,ID,NORMA_DESCRICAO,NORMA_TIPO,NORMA_NUM,NORMA_ANO,EMENTA_DESCRICAO,TEMA_PRINCIPAL,PASTA_PRINCIPAL_FABRIC,PASTA_SECUNDARIA_FABRIC,ARQUIVO,URL,EDITAR,ATUALIZACAO,UNID_RESP_CURADORIA,SERVIDOR_RESP_CURADORIA,OBSERVACAO_NUEIA,OBSERVACAO_AREA_NEGOCIO
0,1,Constituição Federal de 1988,Constituição Federal,<NA>,1988,"Texto constitucional compilado, contendo os pr...",Diversos,Legislacao_Planalto,Constituição Federal,CONSTITUICAO FEDERAL_1988_COMPILADA.txt,https://www.planalto.gov.br/ccivil_03/constitu...,Não,Código Python,NaN,NaN,NaN,NaN
1,2,Decreto-Lei nº 2.848/1940 (Código Penal),Decreto-Lei,2848,1940,"Define os crimes e as penas, incluindo crimes ...",Penal,Legislacao_Planalto,Decreto-Lei,DECRETO-LEI_2848_1940_COMPILADO - CODIGO PENAL...,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
2,3,Decreto-Lei nº 3.689/1941 (Código de Processo ...,Decreto-Lei,3689,1941,Regula o processo penal em todo o território b...,Penal,Legislacao_Planalto,Decreto-Lei,DECRETO-LEI_3689_1941_COMPILADO - CODIGO DE PR...,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
3,4,Decreto-Lei nº 4.657/1942 (LINDB),Decreto-Lei,4657,1942,Lei de Introdução às normas do Direito Brasile...,NaN,Legislacao_Planalto,Decreto-Lei,DECRETO-LEI_4657_1942_COMPILADO - LINDB.txt,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
4,5,Lei nº 10.406/2002 (Código Civil),Lei Ordinária,10406,2002,Institui o Código Civil Brasileiro.,NaN,Legislacao_Planalto,Lei Ordinária,LEI_10406_2002_COMPILADA - CODIGO_CIVIL.txt,https://www.planalto.gov.br/ccivil_03/leis/200...,Não,Código Python,NaN,NaN,NaN,NaN


In [35]:
# Edite a coluna "PASTA_SECUNDARIA_FABRIC" para que o seu conteúdo apresente 
# apenas a primeira letra das palavras em maiúsculo 
df_inventario_fontes['PASTA_SECUNDARIA_FABRIC'] = df_inventario_fontes['PASTA_SECUNDARIA_FABRIC'].apply(limpar_string_title)

# Exiba o DataFrame resultante
df_inventario_fontes.head()

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 37, Finished, Available, Finished, False)

,ID,NORMA_DESCRICAO,NORMA_TIPO,NORMA_NUM,NORMA_ANO,EMENTA_DESCRICAO,TEMA_PRINCIPAL,PASTA_PRINCIPAL_FABRIC,PASTA_SECUNDARIA_FABRIC,ARQUIVO,URL,EDITAR,ATUALIZACAO,UNID_RESP_CURADORIA,SERVIDOR_RESP_CURADORIA,OBSERVACAO_NUEIA,OBSERVACAO_AREA_NEGOCIO
0,1,Constituição Federal de 1988,Constituição Federal,<NA>,1988,"Texto constitucional compilado, contendo os pr...",Diversos,Legislacao_Planalto,Constituicao_Federal,CONSTITUICAO FEDERAL_1988_COMPILADA.txt,https://www.planalto.gov.br/ccivil_03/constitu...,Não,Código Python,NaN,NaN,NaN,NaN
1,2,Decreto-Lei nº 2.848/1940 (Código Penal),Decreto-Lei,2848,1940,"Define os crimes e as penas, incluindo crimes ...",Penal,Legislacao_Planalto,Decreto-Lei,DECRETO-LEI_2848_1940_COMPILADO - CODIGO PENAL...,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
2,3,Decreto-Lei nº 3.689/1941 (Código de Processo ...,Decreto-Lei,3689,1941,Regula o processo penal em todo o território b...,Penal,Legislacao_Planalto,Decreto-Lei,DECRETO-LEI_3689_1941_COMPILADO - CODIGO DE PR...,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
3,4,Decreto-Lei nº 4.657/1942 (LINDB),Decreto-Lei,4657,1942,Lei de Introdução às normas do Direito Brasile...,NaN,Legislacao_Planalto,Decreto-Lei,DECRETO-LEI_4657_1942_COMPILADO - LINDB.txt,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
4,5,Lei nº 10.406/2002 (Código Civil),Lei Ordinária,10406,2002,Institui o Código Civil Brasileiro.,NaN,Legislacao_Planalto,Lei_Ordinaria,LEI_10406_2002_COMPILADA - CODIGO_CIVIL.txt,https://www.planalto.gov.br/ccivil_03/leis/200...,Não,Código Python,NaN,NaN,NaN,NaN


## **3. Criação do DataFrame "Planalto"**

In [36]:
# Crie o DataFrame "Planalto" a partir do DataFrame "Inventário Fontes RAG"
df_planalto = df_inventario_fontes[
    (df_inventario_fontes['PASTA_PRINCIPAL_FABRIC']=='Legislacao_Planalto') &
    (df_inventario_fontes['EDITAR'] != 'Excluir') &
    (df_inventario_fontes['ATUALIZACAO'] == 'Código Python') &
    (df_inventario_fontes['URL'].str.contains(r'\.html?$', case=False, na=False))
]

# Exiba o DataFrame resultante
df_planalto.head()

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 38, Finished, Available, Finished, False)

,ID,NORMA_DESCRICAO,NORMA_TIPO,NORMA_NUM,NORMA_ANO,EMENTA_DESCRICAO,TEMA_PRINCIPAL,PASTA_PRINCIPAL_FABRIC,PASTA_SECUNDARIA_FABRIC,ARQUIVO,URL,EDITAR,ATUALIZACAO,UNID_RESP_CURADORIA,SERVIDOR_RESP_CURADORIA,OBSERVACAO_NUEIA,OBSERVACAO_AREA_NEGOCIO
0,1,Constituição Federal de 1988,Constituição Federal,<NA>,1988,"Texto constitucional compilado, contendo os pr...",Diversos,Legislacao_Planalto,Constituicao_Federal,CONSTITUICAO FEDERAL_1988_COMPILADA.txt,https://www.planalto.gov.br/ccivil_03/constitu...,Não,Código Python,NaN,NaN,NaN,NaN
1,2,Decreto-Lei nº 2.848/1940 (Código Penal),Decreto-Lei,2848,1940,"Define os crimes e as penas, incluindo crimes ...",Penal,Legislacao_Planalto,Decreto-Lei,DECRETO-LEI_2848_1940_COMPILADO - CODIGO PENAL...,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
2,3,Decreto-Lei nº 3.689/1941 (Código de Processo ...,Decreto-Lei,3689,1941,Regula o processo penal em todo o território b...,Penal,Legislacao_Planalto,Decreto-Lei,DECRETO-LEI_3689_1941_COMPILADO - CODIGO DE PR...,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
3,4,Decreto-Lei nº 4.657/1942 (LINDB),Decreto-Lei,4657,1942,Lei de Introdução às normas do Direito Brasile...,NaN,Legislacao_Planalto,Decreto-Lei,DECRETO-LEI_4657_1942_COMPILADO - LINDB.txt,https://www.planalto.gov.br/ccivil_03/decreto-...,Não,Código Python,NaN,NaN,NaN,NaN
4,5,Lei nº 10.406/2002 (Código Civil),Lei Ordinária,10406,2002,Institui o Código Civil Brasileiro.,NaN,Legislacao_Planalto,Lei_Ordinaria,LEI_10406_2002_COMPILADA - CODIGO_CIVIL.txt,https://www.planalto.gov.br/ccivil_03/leis/200...,Não,Código Python,NaN,NaN,NaN,NaN


In [37]:
# Ordene o DataFrame "df_planalto"
df_planalto = df_planalto.sort_values(
    by=['NORMA_TIPO', 'NORMA_ANO', 'NORMA_NUM'],
    ascending=[True, False, False]
)

# Exiba o DataFrame resultante
df_planalto.head()

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 39, Finished, Available, Finished, False)

,ID,NORMA_DESCRICAO,NORMA_TIPO,NORMA_NUM,NORMA_ANO,EMENTA_DESCRICAO,TEMA_PRINCIPAL,PASTA_PRINCIPAL_FABRIC,PASTA_SECUNDARIA_FABRIC,ARQUIVO,URL,EDITAR,ATUALIZACAO,UNID_RESP_CURADORIA,SERVIDOR_RESP_CURADORIA,OBSERVACAO_NUEIA,OBSERVACAO_AREA_NEGOCIO
0,1,Constituição Federal de 1988,Constituição Federal,<NA>,1988,"Texto constitucional compilado, contendo os pr...",Diversos,Legislacao_Planalto,Constituicao_Federal,CONSTITUICAO FEDERAL_1988_COMPILADA.txt,https://www.planalto.gov.br/ccivil_03/constitu...,Não,Código Python,NaN,NaN,NaN,NaN
56,57,Decreto nº 12.174/2024,Decreto,12174,2024,Dispõe sobre as garantias trabalhistas a serem...,Licitações e Contratos,Legislacao_Planalto,Decreto,DECRETO_12174_2024 - GARANTIAS TRABALHISTAS.txt,http://www.planalto.gov.br/ccivil_03/_ato2023-...,Não,Código Python,NaN,NaN,NaN,NaN
55,56,Decreto nº 12.122/2024,Decreto,12122,2024,Institui o Programa Federal de Prevenção e Enf...,Assédio e Discriminação,Legislacao_Planalto,Decreto,DECRETO_12122_2024 - INSTITUI PROGRAMA FEDERAL...,http://www.planalto.gov.br/ccivil_03/_ato2023-...,Não,Código Python,NaN,NaN,NaN,NaN
54,55,Decreto nº 11.907/2024,Decreto,11907,2024,Aprova a Estrutura Regimental e o Quadro Demon...,NaN,Legislacao_Planalto,Decreto,DECRETO_11907_2024 - CARGOS COMISSAO E FC MF.txt,http://www.planalto.gov.br/ccivil_03/_ato2023-...,Não,Código Python,NaN,NaN,NaN,NaN
53,54,Decreto nº 11.529/2023,Decreto,11529,2023,"Institui o Sistema de Integridade, Transparênc...",Integridade,Legislacao_Planalto,Decreto,DECRETO_11529_2023 - SITAI.txt,http://www.planalto.gov.br/ccivil_03/_ato2023-...,Não,Código Python,NaN,NaN,NaN,NaN


## **4. Geração da legislação do Planalto em formato "TXT" e do arquivo "LOG"**

In [38]:
# Gere tanto arquivos em formato TXT a partir do DataFrame "df_planalto" 
# quanto o arquivo de "log"

results = []

# Gera timestamp em horário de Brasília
timestamp_brt = datetime.now(
    ZoneInfo("America/Sao_Paulo")
).strftime("%Y%m%d_%H%M%S")

# Caminho completo do arquivo de log XLSX
log_xlsx = os.path.join(
    log_path,
    f"log_legislacao_planalto_{timestamp_brt}.xlsx"
)

# Garante que a pasta de logs exista
os.makedirs(log_path, exist_ok=True)

# Percorre o DataFrame
for row in df_planalto.dropna(subset=["URL"]).itertuples():

    # -------------------------
    # 1) Diretório dos VIGENTES
    # -------------------------
    output_vigente_dir = os.path.join(
        legislacao_planalto_vigente_path,
        str(row.PASTA_PRINCIPAL_FABRIC).strip(),
        str(row.PASTA_SECUNDARIA_FABRIC).strip()
    )
    os.makedirs(output_vigente_dir, exist_ok=True)

    # -------------------------
    # 2) Diretório dos REVOGADOS    
    # -------------------------
    output_revogado_dir = os.path.join(
        legislacao_planalto_revogada_path,        
        str(row.PASTA_SECUNDARIA_FABRIC).strip()
    )
    # Só cria se for realmente gerar revogado (evita criar árvore desnecessária)
    # Obs.: se você tiver uma coluna do DF indicando isso por linha, pode usar aqui.
    # Como você pediu "caso output_revogado seja True", então está controlado por parâmetro geral.
    os.makedirs(output_revogado_dir, exist_ok=True)

    # -------------------------
    # 3) Gera o TXT da legislação
    # -------------------------
    r = html_planalto_to_txt(
        url=row.URL,
        pasta_secundaria_fabric=row.PASTA_SECUNDARIA_FABRIC,
        arquivo_vigente_nome=row.ARQUIVO,        
        output_vigente_dir=output_vigente_dir,
        log_xlsx_path=log_xlsx,
        output_revogado=True,                 # ✅ agora gera/salva revogado
        output_revogado_dir=output_revogado_dir  
    )

    results.append(r)

# DataFrame de auditoria ao final
df_log = pd.DataFrame(results)

# Exibe o DataFrame resultante
df_log

StatementMeta(, d5ab050a-48d5-4e70-8d42-ac388da2504f, 40, Finished, Available, Finished, False)

,legislacao_planalto_tipo,arquivo_vigente_nome_original,arquivo_vigente_modo,status_qualidade,html_raw_chars,visible_text_chars,txt_vigente_chars,txt_revogado_chars,removed_strike_count,removed_revogado_lines_count,last_art_expected,last_art_extracted,final_url_used
0,Constituicao_Federal,CONSTITUICAO FEDERAL_1988_COMPILADA.txt,SOBRESCRITO,SEM_ALERTA,1500085,649105,648514,667,4,28,250,250,https://www.planalto.gov.br/ccivil_03/constitu...
1,Decreto,DECRETO_12174_2024 - GARANTIAS TRABALHISTAS.txt,SOBRESCRITO,SEM_ALERTA,24979,5381,5371,0,0,0,7,7,http://www.planalto.gov.br/ccivil_03/_ato2023-...
2,Decreto,DECRETO_12122_2024 - INSTITUI PROGRAMA FEDERAL...,SOBRESCRITO,SEM_ALERTA,36017,8585,8563,0,0,0,17,17,http://www.planalto.gov.br/ccivil_03/_ato2023-...
3,Decreto,DECRETO_11907_2024 - CARGOS COMISSAO E FC MF.txt,SOBRESCRITO,ALERTA_EXTRACAO_INCOMPLETA|ALERTA_TERMINOU_CEDO,3873616,183909,176794,55799,3279,0,102,29,http://www.planalto.gov.br/ccivil_03/_ato2023-...
4,Decreto,DECRETO_11529_2023 - SITAI.txt,SOBRESCRITO,SEM_ALERTA,60789,18596,18564,0,0,0,20,20,http://www.planalto.gov.br/ccivil_03/_ato2023-...
5,Decreto,DECRETO_11430_2023 - REGULAMENTA LEI 14133_202...,SOBRESCRITO,SEM_ALERTA,49107,9922,9901,3119,17,0,9,9,http://www.planalto.gov.br/ccivil_03/_ato2023-...
6,Decreto,DECRETO_11129_2022 - REGULAMENTA LAC.txt,SOBRESCRITO,SEM_ALERTA,155230,58375,58270,0,0,0,71,71,http://www.planalto.gov.br/ccivil_03/_ato2019-...
7,Decreto,DECRETO_11123_2022 - DELEGA COMPETENCIA DISCIP...,SOBRESCRITO,ALERTA_EXTRACAO_INCOMPLETA,25280,4360,4347,0,0,0,10,10,http://www.planalto.gov.br/ccivil_03/_ato2019-...
8,Decreto,DECRETO_11072_2022 - PGD.txt,SOBRESCRITO,ALERTA_TERMINOU_CEDO,57150,15855,15817,0,0,0,95,20,http://www.planalto.gov.br/ccivil_03/_ato2019-...
9,Decreto,DECRETO_10929_2022 - PROCEDIMENTO ESPECIAL.txt,SOBRESCRITO,ALERTA_EXTRACAO_INCOMPLETA|ALERTA_TEXTO_CURTO,16156,1683,1678,0,0,0,4,4,http://www.planalto.gov.br/ccivil_03/_ato2019-...


In [19]:
# ============================================================
# TESTE UNITÁRIO (1 URL)
# Objetivo: rodar só uma URL e ver os prints DEBUG do notebook
# ============================================================

# 1) URL problemática
url_teste = "https://www.planalto.gov.br/ccivil_03/codigos/codi_conduta/resolucao7.htm"

# 2) Pasta de teste dentro do Lakehouse (ou CWD)
legislacao_teste_pasta = "teste_legislacao_url"
legislacao_teste_path = base_path / legislacao_teste_pasta

# ✅ cria a pasta NO LUGAR CERTO
legislacao_teste_path.mkdir(parents=True, exist_ok=True)

# 3) Nome base do arquivo
arquivo_nome_teste = "RESOLUCAO_CEP_7_2002 - ATIVIDADES POLITICAS_TESTE"

# 4) Caminho do LOG (dentro da mesma pasta)
log_xlsx_teste = legislacao_teste_path / f"LOG_{arquivo_nome_teste}.xlsx"

print("=== INÍCIO TESTE UNITÁRIO ===")
print("URL:", url_teste)
print("BASE_PATH:", base_path)
print("OUTPUT_DIR:", legislacao_teste_path)
print("LOG_XLSX:", log_xlsx_teste)
print(
    "Timestamp:",
    datetime.now(ZoneInfo("America/Sao_Paulo")).isoformat(timespec="seconds")
)
print("==============================\n")

# 5) Executa a função (dispara os prints DEBUG internos)
r = html_planalto_to_txt(
    url=url_teste,
    pasta_secundaria_fabric="DECRETO",
    arquivo_vigente_nome=arquivo_nome_teste,
    output_vigente_dir=str(legislacao_teste_path),   # <-- Path convertido explicitamente
    output_revogado=True,
    output_revogado_dir=str(legislacao_teste_path),
    log_xlsx_path=str(log_xlsx_teste),
    timeout=(10, 120),
)

print("\n=== FIM TESTE UNITÁRIO ===")
print("RESULT (dict retornado):")
for k, v in r.items():
    print(f"- {k}: {v}")

# 6) Caminhos esperados dos arquivos gerados
vig_path = legislacao_teste_path / f"{arquivo_nome_teste}_vigente.txt"
rev_path = legislacao_teste_path / f"{arquivo_nome_teste}_revogado.txt"

print("\nArquivos esperados:")
print("VIGENTE :", vig_path)
print("REVOGADO:", rev_path)
print("LOG     :", log_xlsx_teste)

StatementMeta(, ce025f58-a5b4-46a3-8a7d-0ffcd5a53414, 21, Finished, Available, Finished, False)

=== INÍCIO TESTE UNITÁRIO ===
URL: https://www.planalto.gov.br/ccivil_03/codigos/codi_conduta/resolucao7.htm
BASE_PATH: /lakehouse/default/Files
OUTPUT_DIR: /lakehouse/default/Files/teste_legislacao_url
LOG_XLSX: /lakehouse/default/Files/teste_legislacao_url/LOG_RESOLUCAO_CEP_7_2002 - ATIVIDADES POLITICAS_TESTE.xlsx
Timestamp: 2026-03-05T10:40:50-03:00


=== FIM TESTE UNITÁRIO ===
RESULT (dict retornado):
- legislacao_planalto_tipo: DECRETO
- arquivo_vigente_nome_original: RESOLUCAO_CEP_7_2002 - ATIVIDADES POLITICAS_TESTE
- arquivo_vigente_modo: CRIADO
- status_qualidade: SEM_ALERTA
- html_raw_chars: 14388
- visible_text_chars: 16454
- txt_vigente_chars: 16467
- txt_revogado_chars: 0
- removed_strike_count: 0
- removed_revogado_lines_count: 0
- last_art_expected: 8
- last_art_extracted: 8
- final_url_used: https://www.planalto.gov.br/ccivil_03/codigos/codi_conduta/resolucao7.htm | http://www.planalto.gov.br/ccivil_03/codigos/codi_conduta/resolucao7.htm

Arquivos esperados:
VIGENTE : /l

In [20]:
# ============================================================
# Checagem rápida: último "Art." no TXT vigente salvo
# ============================================================

# Procura o primeiro arquivo *_vigente.txt dentro do diretório de teste
vigente_path_guess = None

for f in legislacao_teste_path.iterdir():
    if f.is_file() and f.name.lower().endswith("_vigente.txt"):
        vigente_path_guess = f
        break

if vigente_path_guess and vigente_path_guess.exists():
    txt = vigente_path_guess.read_text(encoding="utf-8", errors="replace")

    # Captura apenas cabeçalhos de artigos (começo de linha), para evitar "art. 84" etc.    
    arts = [
        _art_num_to_int(m.group(1))
        for m in re.finditer(rf"(?im)^\s*Art\.\s*({ART_NUM_RE})\b", txt)
    ]
    print("\nChecagem rápida do vigente:")
    print("Arquivo:", str(vigente_path_guess))
    print("Maior Art. (cabeçalho):", max(arts) if arts else None)

    # (Opcional) imprime também o último cabeçalho completo encontrado
    # para facilitar diagnóstico rápido sem abrir o arquivo
    last_hdr = None
    for m in re.finditer(r"(?im)^\s*(Art\.\s*\d+[^\n]*)", txt):
        last_hdr = m.group(1).strip()
    if last_hdr:
        print("Último cabeçalho encontrado:", last_hdr)

else:
    print("\nNão encontrei arquivo *_vigente.txt no diretório de teste:", str(legislacao_teste_path))

StatementMeta(, ce025f58-a5b4-46a3-8a7d-0ffcd5a53414, 22, Finished, Available, Finished, False)


Checagem rápida do vigente:
Arquivo: /lakehouse/default/Files/teste_legislacao_url/RESOLUCAO_CEP_7_2002_-_ATIVIDADES_POLITICAS_TESTE_vigente.txt
Maior Art. (cabeçalho): None
Último cabeçalho encontrado: Art. 8º A Comissão de Ética Pública esclarecerá as dúvidas que eventualmente surjam na efetiva aplicação das normas.
